# Constellation Plot — Naveja & Medina-Franco (2019)

Faithful implementation of the method from:
> Naveja JJ, Medina-Franco JL. *Finding Constellations in Chemical Space Through Core Analysis.*  
> **Front. Chem.** 7:510 (2019). https://doi.org/10.3389/fchem.2019.00510

### Method definitions (Figure 6 of the paper)
| Visual element | Meaning |
|---|---|
| **One circle = one core (scaffold)** | Each point represents a scaffold/core, not a molecule |
| **Circle size** | Relative number of compounds annotated to the core (analog series count) |
| **Circle color** | Mean continuous property of core compounds (e.g.: pIC50, % inhibition) |
| **Lines between circles** | Cores that **share compounds** — forming the "constellations" |
| **Label** | Analog series identifier (AS-N) |
| **2D coordinates** | t-SNE or PCA from fingerprints of core compounds |

## 0. Configuration — edit here before running

In [27]:
#  DATA INPUT

CSV_PATH      = '../Ensemble/all_consensus_zscore.csv'   # << update to your CSV path
SMILES_COL    = 'SMILES'            # SMILES column name
ACTIVITY_COL  = 'consensus_z'          # activity column name (numeric)
ID_COL        = 'ID'            # ID column name (or None)

#  FINGERPRINT AND DIMENSIONALITY REDUCTION

FP_TYPE       = 'ecfp4'  # 'ecfp4' | 'ecfp6' | 'fcfp4'
FP_NBITS      = 1024
DIM_METHOD    = 'tsne'   # 'tsne' | 'pca' | 'umap'

TSNE_PERPLEXITY = 30     # reduce for datasets < 100 compounds
TSNE_ITER       = 1000
RANDOM_STATE    = 42

#  CONNECTION LINES ("CONSTELLATIONS")

# Cores connected if Tanimoto between mean fingerprints >= threshold
TANIMOTO_THRESHOLD = 0.55

# Minimum number of compounds a scaffold must contain to be included in connection calculation
MIN_SCAFFOLD_COUNT = 3    # ≥ 

#  PLOT CONFIG
PLOT_WIDTH      = 900
PLOT_HEIGHT     = 700
OUTPUT_HTML     = '../results/constellation/constellation_plot.html'
OUTPUT_CSV      = '../results/constellation/constellation_cores.csv'

## 1. Imports

In [28]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import base64
from io import BytesIO

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from bokeh.plotting import figure, output_notebook, show, save
from bokeh.models import ColumnDataSource, HoverTool, ColorBar, LinearColorMapper
from bokeh.transform import linear_cmap
from bokeh.palettes import Viridis256
from bokeh.io import output_file
from bokeh import __version__ as bokeh_version

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.DataStructs import ConvertToNumpyArray, BulkTanimotoSimilarity

output_notebook()
print('Imports OK. Bokeh', bokeh_version)

Loading BokehJS ...

Imports OK. Bokeh 3.8.0


## 2. Load and validate data

In [29]:
df_raw = pd.read_csv(CSV_PATH)
print(f'Loaded: {len(df_raw)} rows | columns: {list(df_raw.columns)}')

assert SMILES_COL in df_raw.columns,   f'Column "{SMILES_COL}" not found.'
assert ACTIVITY_COL in df_raw.columns, f'Column "{ACTIVITY_COL}" not found.'

if ID_COL and ID_COL not in df_raw.columns:
    df_raw[ID_COL] = [f'MOL{i+1:04d}' for i in range(len(df_raw))]

df_raw[ACTIVITY_COL] = pd.to_numeric(df_raw[ACTIVITY_COL], errors='coerce')
n_before = len(df_raw)
df_raw = df_raw.dropna(subset=[SMILES_COL, ACTIVITY_COL]).reset_index(drop=True)
print(f'{n_before - len(df_raw)} rows removed (null SMILES/activity). Remaining: {len(df_raw)}')

df_raw.head()

Loaded: 4398 rows | columns: ['ID', 'library', 'SMILES', 'Name', 'Formula', 'MW', 'LogP', 'pKd', 'score', 'z_pkd', 'z_vina', 'consensus_z', 'SA_Score', 'Structure', 'Synthesize', 'Chemical', 'Dataset']
0 rows removed (null SMILES/activity). Remaining: 4398


,ID,library,SMILES,Name,Formula,MW,LogP,pKd,score,z_pkd,z_vina,consensus_z,SA_Score,Structure,Synthesize,Chemical,Dataset
0,Druglike_1,Druglike,NC1=C2C=NNC2=NC([C@@H](/C=C/CC2=NC(CCCl)=CC=C2...,21/000000020,C17H17N6O2Cl,372.0,2.17,6.45,-9.496,6.042542,1.343421,3.692981,3.895024,6.97,24.1,-190.0,DRUGLIKE_CONCAT
1,HighDiv_4,HighDiv,NOC1=CC(C(=O)O)=CC(OC/C=C/[C@@H]2NNC3=C2C=CC=C...,12/000001403,C17H16N3O4Br,406.0,2.98,5.86,-10.609,4.310268,2.570990,3.440629,3.709026,7.55,21.3,-190.0,HIGHDIVERSITY_CONCAT
2,HighDiv_1581,HighDiv,C=CC1=C(C(N)=O)SC2=CC=C([S@+]([O-])C3=CC(O)=C(...,12/000000150,C17H12NO4S2I,485.0,2.13,5.68,-10.932,3.781778,2.927239,3.354509,3.880718,6.18,24.8,-40.0,HIGHDIVERSITY_LANAPDB
3,HighDiv_1,HighDiv,C[C@H](CN)CC#CC1=CC(C(=O)CO)=CC(N=O)=C1Br,12/000000691,C14H16N2O3Br,340.0,1.76,6.10,-9.375,5.014922,1.209965,3.112444,4.386506,5.24,20.7,-170.0,HIGHDIVERSITY_CONCAT
4,Druglike_2,Druglike,NC1=C2C=NNC2=NC([C@@H](/C=C/CC2=NC=CC=C2CCBr)C...,21/000000030,C17H17N6O2Br,417.0,2.42,6.04,-9.526,4.838759,1.376509,3.107634,3.978020,5.01,24.1,-190.0,DRUGLIKE_CONCAT


## 3. Compute fingerprints and Murcko scaffold

In [30]:
def smiles_to_fp(smi, fp_type='ecfp4', nbits=2048):
    """Returns (mol, BitVect, np.array) or (None, None, None) if invalid."""
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None, None, None
    radius = {'ecfp4': 2, 'ecfp6': 3, 'fcfp4': 2}[fp_type]
    use_feat = fp_type.startswith('fcfp')
    bv  = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits, useFeatures=use_feat)
    arr = np.zeros(nbits, dtype=np.uint8)
    ConvertToNumpyArray(bv, arr)
    return mol, bv, arr


records, skipped = [], []

for _, row in df_raw.iterrows():
    mol, bv, arr = smiles_to_fp(row[SMILES_COL], FP_TYPE, FP_NBITS)
    if mol is None:
        skipped.append(row[SMILES_COL])
        continue

    try:
        core = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        if not core:
            core = Chem.MolToSmiles(mol)   # acyclic: use molecule's own SMILES
    except Exception:
        core = Chem.MolToSmiles(mol)

    records.append({
        'mol_id':   row.get(ID_COL, f'MOL{len(records)+1:04d}'),
        'smiles':   row[SMILES_COL],
        'core':     core,
        'activity': float(row[ACTIVITY_COL]),
        'fp_bv':    bv,
        'fp_arr':   arr,
    })

df = pd.DataFrame(records)
X  = np.vstack(df['fp_arr'].values)

print(f'{len(df)} valid molecules | {len(skipped)} discarded')
print(f'{df["core"].nunique()} unique cores (Murcko)')
df[['mol_id', 'core', 'activity']].head()

4398 valid molecules | 0 discarded
1231 unique cores (Murcko)


[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerator
[12:36:40] DEPRECATION WARNING: please use MorganGenerat

,mol_id,core,activity
0,Druglike_1,C(=CCc1ncc2cn[nH]c2n1)Cc1ccccn1,3.692981
1,HighDiv_4,C(=CC1NNc2ccccc21)COc1ccccc1,3.440629
2,HighDiv_1581,c1ccc([SH+]c2ccc3sccc3c2)cc1,3.354509
3,HighDiv_1,c1ccccc1,3.112444
4,Druglike_2,C(=CCc1ncc2cn[nH]c2n1)Cc1ccccn1,3.107634


## 4. Dimensionality reduction + core construction

In [31]:
# ── Dimensionality reduction ────────────────────────────────────────
if DIM_METHOD == 'tsne':
    perp   = min(TSNE_PERPLEXITY, len(df) - 1)
    coords = TSNE(n_components=2, perplexity=perp,
                  n_iter=TSNE_ITER, random_state=RANDOM_STATE).fit_transform(X)
    x_label, y_label = 't-SNE 1', 't-SNE 2'

elif DIM_METHOD == 'pca':
    pca    = PCA(n_components=2, random_state=RANDOM_STATE)
    coords = pca.fit_transform(X)
    var    = pca.explained_variance_ratio_ * 100
    x_label, y_label = f'PC1 ({var[0]:.1f}%)', f'PC2 ({var[1]:.1f}%)'

elif DIM_METHOD == 'umap':
    import umap
    coords = umap.UMAP(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
    x_label, y_label = 'UMAP 1', 'UMAP 2'

df['x'], df['y'] = coords[:, 0], coords[:, 1]

# ── Aggregate: one point per core ──────────────────────────────────
cores = (
    df.groupby('core')
    .agg(
        x_mean        = ('x', 'mean'),
        y_mean        = ('y', 'mean'),
        count         = ('mol_id', 'count'),
        mean_activity = ('activity', 'mean'),
        mol_ids       = ('mol_id', lambda s: ', '.join(s.tolist())),
    )
    .reset_index()
    .rename(columns={'core': 'core_smiles'})
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

# Mean fingerprint per core (for connection calculation)
def mean_fp_for_core(core_smi):
    return df.loc[df['core'] == core_smi, 'fp_arr'].values.mean(axis=0)

cores['fp_mean'] = cores['core_smiles'].apply(mean_fp_for_core)

print(f'Dimensionality computed. {len(cores)} total cores.')

# Filter: only plot scaffolds with >= MIN_SCAFFOLD_COUNT compounds
n_before_filter = len(cores)
cores = cores[cores['count'] >= MIN_SCAFFOLD_COUNT].reset_index(drop=True)
cores['series_id'] = [f'AS-{i+1}' for i in range(len(cores))]
print(f'{n_before_filter - len(cores)} scaffolds excluded from plot (< {MIN_SCAFFOLD_COUNT} compounds). Remaining for plot: {len(cores)}')

cores[['series_id', 'count', 'mean_activity', 'x_mean', 'y_mean']].head(10)

Dimensionality computed. 1231 total cores.
1018 scaffolds excluded from plot (< 3 compounds). Remaining for plot: 213


,series_id,count,mean_activity,x_mean,y_mean
0,AS-1,941,-0.358063,29.977718,-5.013032
1,AS-2,146,0.034071,-6.942873,72.595894
2,AS-3,136,-0.555221,27.317528,28.759205
3,AS-4,101,1.477173,-80.251167,-11.961108
4,AS-5,91,-0.195928,-11.432337,34.220695
5,AS-6,84,0.588237,-3.318406,-78.066986
6,AS-7,60,-0.619439,-29.194204,12.103948
7,AS-8,57,-0.001644,-54.455109,-25.185930
8,AS-9,51,0.266754,-26.271875,-24.619394
9,AS-10,51,-1.000125,-54.429638,22.326248


## 5. Compute connections between cores (the "constellations")

In the original paper, cores are connected when they **share compounds** (analog series overlap).  
Since the Murcko scaffold is unique per compound, we use **Tanimoto between mean core fingerprints** as a proxy.

In [32]:
def arr_to_bv(arr):
    """Converts mean float array to ExplicitBitVect (binarized at 0.5)."""
    bv = DataStructs.ExplicitBitVect(len(arr))
    for i, v in enumerate(arr):
        if v >= 0.5:
            bv.SetBit(i)
    return bv

# cores already filtered by MIN_SCAFFOLD_COUNT in previous cell
cores_for_edges = cores
print(f'{len(cores_for_edges)} scaffolds with >= {MIN_SCAFFOLD_COUNT} compounds (computing connections)')

bvs   = [arr_to_bv(fp) for fp in cores_for_edges['fp_mean'].values]
edges = []

for i, bv_i in enumerate(bvs):
    sims = BulkTanimotoSimilarity(bv_i, bvs)
    for j, sim in enumerate(sims):
        if j > i and sim >= TANIMOTO_THRESHOLD:
            edges.append((
                cores_for_edges.index[i],
                cores_for_edges.index[j],
                float(sim)
            ))

print(f'{len(edges)} connections (Tanimoto >= {TANIMOTO_THRESHOLD})')

213 scaffolds with >= 3 compounds (computing connections)
71 connections (Tanimoto >= 0.55)


## 6. Interactive Constellation Plot (Bokeh)

In [33]:
# ── Generate base64 scaffold images for hover ─────────────────────
def smiles_to_base64_img(smi, size=(220, 160)):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return ''
    img = Draw.MolToImage(mol, size=size)
    buf = BytesIO()
    img.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode('utf-8')

print('Generating scaffold images...')
cores['img_b64'] = cores['core_smiles'].apply(smiles_to_base64_img)
print(f'{len(cores)} images generated.')

# ── Normalize point sizes: sqrt-scaled, mapped to [5, 30] px ──────
sqrt_vals = np.sqrt(cores['count'].values.astype(float))
s_min, s_max = 4, 65
if sqrt_vals.max() > sqrt_vals.min():
    sizes = s_min + (s_max - s_min) * (sqrt_vals - sqrt_vals.min()) / (sqrt_vals.max() - sqrt_vals.min())
else:
    sizes = np.full(len(cores), (s_min + s_max) / 2)

# ── Color mapper (Viridis — perceptually uniform) ──────────────────
act_min = float(cores['mean_activity'].min())
act_max = float(cores['mean_activity'].max())
mapper  = LinearColorMapper(palette=Viridis256, low=act_min, high=act_max)

# ── ColumnDataSource ───────────────────────────────────────────────
source = ColumnDataSource(dict(
    x           = cores['x_mean'].tolist(),
    y           = cores['y_mean'].tolist(),
    series_id   = cores['series_id'].tolist(),
    count       = cores['count'].tolist(),
    mean_act    = cores['mean_activity'].round(3).tolist(),
    core_smiles = cores['core_smiles'].tolist(),
    mol_ids     = cores['mol_ids'].tolist(),
    size        = sizes.tolist(),
    img_b64     = cores['img_b64'].tolist(),
    act_color   = cores['mean_activity'].tolist(),
))

# ── Figure ─────────────────────────────────────────────────────────
p = figure(
    width=PLOT_WIDTH,
    height=PLOT_HEIGHT,
    title=(
        f'Constellation Plot — {FP_TYPE.upper()} / {DIM_METHOD.upper()}'
    ),
    x_axis_label=x_label,
    y_axis_label=y_label,
    toolbar_location='above',
    tools='pan,wheel_zoom,box_zoom,reset,save',
)
p.background_fill_color = 'white'
p.grid.grid_line_color  = '#dddddd'

# ── Layer 1: connection lines ──────────────────────────────────────
if edges:
    p.segment(
        x0=[cores.loc[i, 'x_mean'] for (i, j, _) in edges],
        y0=[cores.loc[i, 'y_mean'] for (i, j, _) in edges],
        x1=[cores.loc[j, 'x_mean'] for (i, j, _) in edges],
        y1=[cores.loc[j, 'y_mean'] for (i, j, _) in edges],
        color='#8c8c8c', alpha=0.35, line_width=0.8,
    )

# ── Layer 2: scaffold circles ──────────────────────────────────────
circles = p.scatter(
    x='x', y='y',
    size='size',
    marker='circle',
    source=source,
    fill_color=linear_cmap('act_color', Viridis256, act_min, act_max),
    fill_alpha=0.90,
    line_color='#404040',
    line_alpha=0.45,
    line_width=0.8,
)

# ── ColorBar ───────────────────────────────────────────────────────
p.add_layout(
    ColorBar(color_mapper=mapper, label_standoff=8, width=14,
             location=(0, 0), title=ACTIVITY_COL),
    'right',
)

# ── HoverTool with scaffold image ──────────────────────────────────
hover = HoverTool(
    renderers=[circles],
    tooltips=f"""
    <div style="padding:8px; background:white; border:1px solid #ccc;
                border-radius:6px; font-family:Arial,sans-serif; font-size:12px;">
        <b style="font-size:13px;">@series_id</b><br>
        <span>Compounds: <b>@count</b></span><br>
        <span>{ACTIVITY_COL} (mean): <b>@mean_act</b></span><br>
        <span style="font-size:10px; color:#666;">Scaffold: @core_smiles</span><br>
        <img src="data:image/png;base64,@img_b64"
             style="margin-top:6px; width:220px; height:160px;
                    border:1px solid #eee; border-radius:3px; display:block;">
    </div>
    """,
)
p.add_tools(hover)

show(p)

Generating scaffold images...
213 images generated.


## 7. Export results

In [34]:
export_cols = ['series_id', 'core_smiles', 'count', 'mean_activity', 'x_mean', 'y_mean', 'mol_ids']
cores[export_cols].to_csv(OUTPUT_CSV, index=False)
print(f'Core table saved: {OUTPUT_CSV}')

output_file(OUTPUT_HTML)
save(p)
print(f'Interactive plot saved: {OUTPUT_HTML}')

Core table saved: ../results/constellation/constellation_cores.csv
Interactive plot saved: ../results/constellation/constellation_plot.html
